<a href="https://colab.research.google.com/github/Heng1222/VeriPromiseESG_2026_TEAM_9906/blob/feat-model-train/app/model/model_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from huggingface_hub import HfApi, login, create_repo
import os
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import gc
import os
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup, BertTokenizerFast
from sklearn.metrics import f1_score
import datetime
from huggingface_hub import hf_hub_download

# ==========================================
# 0. 全域設定與超參數
# ==========================================
MODEL_NAME = "ckiplab/bert-base-chinese"
MAX_LEN = 512
BATCH_SIZE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ==========================================
# 1. 資料前處理與 Dataset (包含 Head+Tail 截斷)
# ==========================================
class ESG_MTL_Dataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512, is_test=False):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

        self.head_len = int((max_len - 2) * 0.25)
        self.tail_len = (max_len - 2) - self.head_len

        # 標籤映射
        self.t1_map = {"No": 0, "Yes": 1}
        self.t2_map = {"already": 0, "within_2_years": 1, "between_2_and_5_years": 2, "longer_than_5_years": 3, "more_than_5_years": 3}
        self.t3_map = {"No": 0, "Yes": 1}
        self.t4_map = {"Clear": 0, "Not Clear": 1, "Misleading": 2}

    def _process_esg_type(self, esg_val):
        if pd.isna(esg_val) or str(esg_val).strip() == "": return "[ESG_UNK]"
        return " ".join([f"[ESG_{t.strip()}]" for t in str(esg_val).split(';')])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. 文本拼接
        esg_prefix = self._process_esg_type(row.get('esg_type', ''))
        raw_text = str(row['data'])
        full_text = f"{esg_prefix} 文本內容：{raw_text}"

        # 2. Token-level 雙向截斷
        tokens = self.tokenizer.encode(full_text, add_special_tokens=False)
        if len(tokens) > (self.max_len - 2):
            tokens = tokens[:self.head_len] + tokens[-self.tail_len:]

        input_ids = [self.tokenizer.cls_token_id] + tokens + [self.tokenizer.sep_token_id]
        attention_mask = [1] * len(input_ids)

        # 3. Padding
        pad_len = self.max_len - len(input_ids)
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        attention_mask += [0] * pad_len

        item = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long)
        }

        # 推論模式不回傳 Labels
        if self.is_test: return item

        # 4. 標籤轉換 (若為 N/A 或缺失，設為 -1 供 Loss 函數忽略)
        item['t1_label'] = torch.tensor(self.t1_map.get(str(row.get('promise_status')), -1), dtype=torch.float)
        item['t2_label'] = torch.tensor(self.t2_map.get(str(row.get('verification_timeline')), -1), dtype=torch.long)
        item['t3_label'] = torch.tensor(self.t3_map.get(str(row.get('evidence_status')), -1), dtype=torch.float)
        item['t4_label'] = torch.tensor(self.t4_map.get(str(row.get('evidence_quality')), -1), dtype=torch.long)

        return item

# ==========================================
# 2. 統一多任務模型架構 (MTL Backbone + 4 Heads)
# ==========================================
class ESG_Unified_MTL_Model(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size

        # T1 & T3 (二元分類): Multi-Sample Dropout 防禦過擬合
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])
        self.t1_head = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1)
        )
        self.t3_head = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1)
        )
        # T2 (4分類) & T4 (3分類)
        self.t2_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 4)
        )
        self.t4_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 3)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :] # 提取 [CLS] 向量

        # Multi-Sample Dropout 平均化
        t1_logits = self.t1_head(cls_output).squeeze(-1)
        t3_logits = self.t3_head(cls_output).squeeze(-1)

        t2_logits = self.t2_head(cls_output)
        t4_logits = self.t4_head(cls_output)

        return t1_logits, t2_logits, t3_logits, t4_logits


def ensemble_inference_and_export(repo_id, test_csv_path, output_csv_path="output.csv"):
    print("🚀 啟動 Soft-Voting 記憶體置換集成推論管線...")

    # 1. 讀取測試資料與 Dataset 初始化
    # 假設測試集有 'id', 'data', 'esg_type' 欄位
    test_df = pd.read_csv(test_csv_path)

    # tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer = BertTokenizerFast.from_pretrained(
      repo_id,
      subfolder="tokenizer" # 告訴 API 去這個 repo 底下的這個資料夾找檔案
    )
    special_tokens = {'additional_special_tokens': ['[ESG_E]', '[ESG_S]', '[ESG_G]', '[ESG_UNK]']}
    tokenizer.add_special_tokens(special_tokens)

    # 注意：is_test=True 讓 Dataset 不會去尋找 label 欄位
    test_dataset = ESG_MTL_Dataset(test_df, tokenizer, MAX_LEN, is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    num_samples = len(test_df)

    # 2. 初始化 Logits 累加器 (Numpy Arrays 存於 CPU RAM，節省 VRAM)
    acc_t1_logits = np.zeros(num_samples)
    acc_t2_logits = np.zeros((num_samples, 4))
    acc_t3_logits = np.zeros(num_samples)
    acc_t4_logits = np.zeros((num_samples, 3))

    # 3. 記憶體置換迴圈：逐一載入 5 個 Fold 的模型
    for fold in range(1, 6):
        print(f"\n📂 載入 Fold {fold} 專家權重...")
        model_path = hf_hub_download(
            repo_id=repo_id,
            filename=f"best_mtl_model_fold_{fold}.pth"
        )
        model = ESG_Unified_MTL_Model(MODEL_NAME)
        model.backbone.resize_token_embeddings(len(tokenizer))

        # 載入下載的權重
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        model.to(DEVICE)
        model.eval()

        fold_t1, fold_t2, fold_t3, fold_t4 = [], [], [], []

        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Fold {fold} 推論中"):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)

                # 取得該批次的 Logits
                t1_log, t2_log, t3_log, t4_log = model(input_ids, attention_mask)

                fold_t1.append(t1_log.cpu().numpy())
                fold_t2.append(t2_log.cpu().numpy())
                fold_t3.append(t3_log.cpu().numpy())
                fold_t4.append(t4_log.cpu().numpy())

        # 將批次結果拼接並累加到全域累加器中
        acc_t1_logits += np.concatenate(fold_t1, axis=0)
        acc_t2_logits += np.concatenate(fold_t2, axis=0)
        acc_t3_logits += np.concatenate(fold_t3, axis=0)
        acc_t4_logits += np.concatenate(fold_t4, axis=0)

        # 核心防禦：強制釋放 VRAM，防止 CUDA OOM
        del model
        gc.collect()
        torch.cuda.empty_cache()

    # 4. 軟投票平均 (Soft-Voting Averaging)
    print("\n🧠 執行 Logits 平均與機率映射...")
    avg_t1_logits = acc_t1_logits / 5.0
    avg_t2_logits = acc_t2_logits / 5.0
    avg_t3_logits = acc_t3_logits / 5.0
    avg_t4_logits = acc_t4_logits / 5.0

    # 5. 激勵函數與最終決策邊界 (Decision Boundary)
    # T1 & T3 轉 Sigmoid 機率；T2 & T4 找最大機率的索引
    # 這裡預設二元分類的閥值(Threshold)為 0.5，後續可透過驗證集最佳化
    t1_preds = (1 / (1 + np.exp(-avg_t1_logits)) > 0.5).astype(int)
    t3_preds = (1 / (1 + np.exp(-avg_t3_logits)) > 0.5).astype(int)
    t2_preds = np.argmax(avg_t2_logits, axis=1)
    t4_preds = np.argmax(avg_t4_logits, axis=1)

    # 6. 逆向對齊字典 (Inverse Mapping)
    inv_t1 = {0: "No", 1: "Yes"}
    inv_t2 = {0: "already", 1: "within_2_years", 2: "between_2_and_5_years", 3: "longer_than_5_years"}
    inv_t3 = {0: "No", 1: "Yes"}
    inv_t4 = {0: "Clear", 1: "Not Clear", 2: "Misleading"}

    # 7. 嚴格路由管線 (Strict Routing Pipeline)
    # 第一性原理：如果上游預測為 No，下游必須強制為 N/A，否則會被競賽平台判定為格式錯誤
    results = []
    for i in range(num_samples):
        if t1_preds[i] == 0:
            # 任務一判定無承諾，後續全數截斷
            results.append({
                "id": test_df.iloc[i]['id'],
                "promise_status": "No",
                "verification_timeline": "N/A",
                "evidence_status": "N/A",
                "evidence_quality": "N/A"
            })
        else:
            # 任務一有承諾，繼續解析時間軸與證據
            t2_res = inv_t2[t2_preds[i]]
            if t3_preds[i] == 0:
                results.append({
                    "id": test_df.iloc[i]['id'],
                    "promise_status": "Yes",
                    "verification_timeline": t2_res,
                    "evidence_status": "No",
                    "evidence_quality": "N/A"
                })
            else:
                results.append({
                    "id": test_df.iloc[i]['id'],
                    "promise_status": "Yes",
                    "verification_timeline": t2_res,
                    "evidence_status": "Yes",
                    "evidence_quality": inv_t4[t4_preds[i]]
                })

    # 8. 輸出最終結果
    final_output = pd.DataFrame(results)
    final_output.to_csv(output_csv_path, index=False)
    print(f"🎉 推論完成！高精度預測結果已匯出至: {output_csv_path}")
    print(final_output.head())

# ==========================================
# 啟動指令範例：
# 以fold_1的val data當測試
test_url = f"https://raw.githubusercontent.com/Heng1222/VeriPromiseESG_2026_TEAM_9906/main/app/data/clean_data/val_fold_1.csv"
ensemble_inference_and_export(f"maxbeettww/VeriPromise_ESG_2026_9906", test_url, "final_submission.csv")

# ==========================================

🚀 啟動 Soft-Voting 記憶體置換集成推論管線...


tokenizer_config.json:   0%|          | 0.00/417 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


📂 載入 Fold 1 專家權重...


best_mtl_model_fold_1.pth:   0%|          | 0.00/409M [00:00<?, ?B/s]

...


📂 載入 Fold 5 專家權重...


best_mtl_model_fold_5.pth:   0%|          | 0.00/409M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstr


🧠 執行 Logits 平均與機率映射...
🎉 推論完成！高精度預測結果已匯出至: final_submission.csv
      id promise_status  verification_timeline evidence_status  \
0  10003            Yes  between_2_and_5_years             Yes   
1  10010            Yes                already             Yes   
2  10018            Yes    longer_than_5_years             Yes   
3  10021            Yes                already             Yes   
4  10027            Yes                already             Yes   

  evidence_quality  
0        Not Clear  
1            Clear  
2            Clear  
3            Clear  
4            Clear  
